In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

# Cargar datos del .env
load_dotenv('../.env')
user = os.getenv('DB_USER')
password = os.getenv('MI_CONTRA')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
database = os.getenv('DB_NAME')

# Crear conexion
engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{database}')

# Probar que funciona
with engine.connect() as conn:
    print("Conexion exitosa a PostgreSQL")

Conexion exitosa a PostgreSQL


In [2]:
# Cargar CSVs del paso anterior
fact_ventas = pd.read_csv('../data/processed/fact_ventas.csv')
dim_producto = pd.read_csv('../data/processed/dim_producto.csv')
dim_cliente = pd.read_csv('../data/processed/dim_cliente.csv')
dim_tienda = pd.read_csv('../data/processed/dim_tienda.csv')

print(f"fact_ventas:  {fact_ventas.shape[0]} filas")
print(f"dim_producto: {dim_producto.shape[0]} filas")
print(f"dim_cliente:  {dim_cliente.shape[0]} filas")
print(f"dim_tienda:   {dim_tienda.shape[0]} filas")

fact_ventas:  5993 filas
dim_producto: 26 filas
dim_cliente:  2052 filas
dim_tienda:   43 filas


In [3]:
# Crear tablas con PK y FK
with engine.connect() as conn:
    # Eliminar tablas si existen
    conn.execute(text("DROP TABLE IF EXISTS fact_ventas CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_producto CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_cliente CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_tienda CASCADE"))
    conn.commit()
    
    # Crear dim_producto (PK: id_producto)
    conn.execute(text("""
        CREATE TABLE dim_producto (
            id_producto VARCHAR(10) PRIMARY KEY,
            nombre_producto VARCHAR(100),
            categoria VARCHAR(50),
            tipo_prenda VARCHAR(50)
        )
    """))
    
    # Crear dim_cliente (PK simple)
    conn.execute(text("""
        CREATE TABLE dim_cliente (
            id_cliente VARCHAR(10) PRIMARY KEY,
            genero_cliente VARCHAR(20),
            tipo_cliente VARCHAR(20)
        )
    """))
    
    # Crear dim_tienda (PK simple)
    conn.execute(text("""
        CREATE TABLE dim_tienda (
            id_tienda VARCHAR(10) PRIMARY KEY,
            nombre_tienda VARCHAR(100),
            ciudad VARCHAR(50),
            departamento VARCHAR(50),
            empresa VARCHAR(50)
        )
    """))
    
    # Crear fact_ventas (FKs a dim_cliente y dim_tienda)
    conn.execute(text("""
        CREATE TABLE fact_ventas (
            id_venta VARCHAR(10),
            numero_linea INT,
            fecha_venta DATE,
            id_cliente VARCHAR(10),
            id_producto VARCHAR(10),
            id_tienda VARCHAR(10),
            cantidad INT,
            precio_unitario DECIMAL(12,2),
            descuento DECIMAL(5,2),
            metodo_pago VARCHAR(50),
            canal VARCHAR(50),
            temporada VARCHAR(50),
            talla VARCHAR(20),
            PRIMARY KEY (id_venta, numero_linea),
            FOREIGN KEY (id_cliente) REFERENCES dim_cliente(id_cliente),
            FOREIGN KEY (id_tienda) REFERENCES dim_tienda(id_tienda),
            FOREIGN KEY (id_producto) REFERENCES dim_producto(id_producto)
        )
    """))
    conn.commit()

In [4]:
# Insertar datos (dimensiones primero, fact despues)
dim_producto.to_sql('dim_producto', engine, if_exists='append', index=False)
print("dim_producto insertada")

dim_cliente.to_sql('dim_cliente', engine, if_exists='append', index=False)
print("dim_cliente insertada")

dim_tienda.to_sql('dim_tienda', engine, if_exists='append', index=False)
print("dim_tienda insertada")

fact_ventas.to_sql('fact_ventas', engine, if_exists='append', index=False)
print("fact_ventas insertada")

print("\nTodas las tablas cargadas correctamente")

dim_producto insertada
dim_cliente insertada
dim_tienda insertada
fact_ventas insertada

Todas las tablas cargadas correctamente


In [5]:
# Consulta de prueba
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            t.empresa,
            COUNT(*) as total_ventas,
            SUM(f.cantidad) as unidades_vendidas,
            ROUND(SUM(f.precio_unitario * f.cantidad * (1 - f.descuento))::numeric, 2) as ingreso_total
        FROM fact_ventas f
        JOIN dim_tienda t ON f.id_tienda = t.id_tienda
        GROUP BY t.empresa
        ORDER BY ingreso_total DESC
    """))
    
    print("VENTAS POR EMPRESA")
    for row in result:
        print(f"  {row[0]}: {row[1]} ventas, {row[2]} unidades, ${row[3]:,.2f}")

VENTAS POR EMPRESA
  KOAJ: 5993 ventas, 11255 unidades, $1,368,581,950.00
